# Etapa 2B — Rastreabilidade persistente
Cria a tabela `medallion_dev_controle.execucoes_eventos` e o Volume `medallion_dev_controle.evidencias`.

O teste registra duas tentativas bem-sucedidas do mesmo lote e uma **falha deliberada**. O resultado esperado é SUCESSO, SUCESSO, FALHOU. Falhas de armazenamento não são ignoradas.

Guarde o valor de RUN_IDS_PARA_RELEITURA para o notebook 03. As evidências JSON já ficam no Volume; o output visual é apenas um resumo. A criação dos objetos também valida as permissões necessárias.

**Parâmetros:** execute apenas a primeira célula de código para exibir os widgets. Preencha os campos e depois execute as demais células. Use **Run all** quando os parâmetros já estiverem preenchidos.


In [ ]:
# 1. Exibir parâmetros — execute esta célula antes de preencher os campos.
dbutils.widgets.text("catalogo", spark.catalog.currentCatalog())
dbutils.widgets.text("batch_id", "")

print("Preencha os widgets acima e depois execute as próximas células.")


In [ ]:
from pathlib import Path
import sys
atual = Path.cwd().resolve()
raiz = next((p for p in [atual, *atual.parents]
             if (p / "src" / "medallion").is_dir() and (p / "pyproject.toml").is_file()), None)
if raiz is None:
    raise RuntimeError("Abra este notebook na pasta Git; se necessário, use %cd com a raiz do projeto.")
if str(raiz / "src") not in sys.path:
    sys.path.insert(0, str(raiz / "src"))
from medallion.config import PipelineConfig



In [ ]:
from medallion.observability.proof import executar_prova

params = {"ambiente": "dev", "catalogo": dbutils.widgets.get("catalogo").strip()}
if dbutils.widgets.get("batch_id").strip():
    params["batch_id"] = dbutils.widgets.get("batch_id").strip()
config = PipelineConfig(**params)
relatorio = executar_prova(spark, config, raiz)
for tentativa in relatorio:
    print(tentativa)

